# 10 - Mini Project: Train And Explain A Tiny Classifier

这一节做一个收尾小项目。

这一节只抓一句话：

> 你要能从数据开始，完整跑通预测、loss、backward、更新、评估，并解释每一步为什么存在。

这不是新知识，而是把 00-09 串成一个作品。

<!-- xiao-preview:start -->
## 课前预习作业：先数项目的输入输出

先看一个小数据集，确认输入维度、输出维度和样本数。

In [ ]:
project_xs = [
    [-2.0, -1.0],
    [-1.0, -1.0],
    [-1.0,  2.0],
    [ 1.0,  1.0],
    [ 2.0, -1.0],
    [ 2.0,  2.0],
]
project_ys = [-1, -1, 1, 1, 1, 1]

# TODO: 把 None 改成你的答案。
preview_num_samples = None
preview_input_dim = None
preview_output_dim = None


def qa_check_10_preview(num_samples, input_dim, output_dim):
    if num_samples is None or input_dim is None or output_dim is None:
        print('请先填写 TODO：先数样本数、输入维度、输出维度。')
        return
    assert num_samples == 6, f'样本数应该是 6，但你填的是 {num_samples}'
    assert input_dim == 2, f'每个输入有 2 个数字，但你填的是 {input_dim}'
    assert output_dim == 1, f'每个样本输出 1 个 score/label，但你填的是 {output_dim}'
    print('OK: 小项目预习通过。')


qa_check_10_preview(preview_num_samples, preview_input_dim, preview_output_dim)

<details>
<summary><strong>Show / Hide 提示</strong></summary>

数 `project_xs` 有几行；每行有几个数字；`project_ys` 每个样本只有一个标签。

</details>

<details>
<summary><strong>Show / Hide 答案</strong></summary>

```python
preview_num_samples = 6
preview_input_dim = 2
preview_output_dim = 1
```

</details>

## 0. Setup

In [ ]:
from pathlib import Path
import sys
import random

cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent, cwd / 'micrograd', cwd.parent / 'micrograd']
PROJECT_ROOT = None
for candidate in candidates:
    if (candidate / 'micrograd' / 'engine.py').exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise RuntimeError(f'Could not find local micrograd package from {cwd}')

sys.path.insert(0, str(PROJECT_ROOT))

from micrograd.engine import Value
from micrograd.nn import MLP

random.seed(1337)
print('project root:', PROJECT_ROOT)

try:
    import matplotlib.pyplot as plt
    MATPLOTLIB_AVAILABLE = True
except Exception as exc:
    MATPLOTLIB_AVAILABLE = False
    print('matplotlib unavailable:', exc)

## 1. 项目地图

| 步骤 | 你要产出什么 |
|---|---|
| 数据 | `xs`, `ys` |
| 模型 | `MLP(2, [4, 1])` |
| 预测 | 每个样本一个 score |
| loss | 平方误差或 hinge loss |
| 训练 | 多轮 zero_grad/backward/update |
| 评估 | accuracy 和最终 loss |
| 解释 | 能说清每一步的作用 |

## 2. 数据和模型

In [ ]:
xs = project_xs
ys = project_ys

random.seed(42)
model = MLP(2, [4, 1])
print(model)
print('parameters:', len(model.parameters()))

for x, y in zip(xs, ys):
    print(x, '->', y)

## 3. 写 predict、loss、accuracy

这个小项目用平方误差：

$$
loss = 
rac{1}{N}\sum_i (score_i - y_i)^2
$$

分类时再看：

```text
score > 0 -> 预测 1
score < 0 -> 预测 -1
```

In [ ]:
def predict(xrow):
    return model([Value(x) for x in xrow])


def loss():
    scores = [predict(x) for x in xs]
    losses = [(score - y) ** 2 for score, y in zip(scores, ys)]
    total = sum(losses) * (1.0 / len(losses))
    return total, scores


def label_from_score(score):
    return 1 if score > 0 else -1


def accuracy(scores, labels):
    preds = [label_from_score(score.data) for score in scores]
    correct = sum(int(pred == y) for pred, y in zip(preds, labels))
    return correct / len(labels), preds

initial_loss, initial_scores = loss()
initial_acc, initial_preds = accuracy(initial_scores, ys)
print('initial loss:', initial_loss.data)
print('initial acc :', initial_acc)
print('initial preds:', initial_preds)

## 4. 训练小项目

这里训练 100 步。你要观察三件事：

```text
loss 是否下降
accuracy 是否提升
predictions 是否更接近标签
```

In [ ]:
learning_rate = 0.05
history = []

for step in range(100):
    total_loss, scores = loss()
    acc, preds = accuracy(scores, ys)
    history.append(total_loss.data)

    model.zero_grad()
    total_loss.backward()

    for p in model.parameters():
        p.data += -learning_rate * p.grad

    if step % 20 == 0 or step == 99:
        print(f'step {step:03d} loss={total_loss.data:.5f} acc={acc:.2f} preds={preds}')

final_loss, final_scores = loss()
final_acc, final_preds = accuracy(final_scores, ys)
print()
print('initial loss:', history[0])
print('final loss  :', final_loss.data)
print('final acc   :', final_acc)
print('final preds :', final_preds)
assert final_loss.data < history[0]

## 5. 看结果，不只看数字

如果能画图，就画出训练后的分界区域。这个图是给直觉用的：蓝色区域预测 `1`，红色区域预测 `-1`。

In [ ]:
if MATPLOTLIB_AVAILABLE:
    grid_x = [i / 10 for i in range(-30, 31)]
    grid_y = [i / 10 for i in range(-30, 31)]
    Z = []
    for yy in grid_y:
        row = []
        for xx in grid_x:
            row.append(predict([xx, yy]).data)
        Z.append(row)

    plt.figure(figsize=(5, 4))
    plt.contourf(grid_x, grid_y, Z, levels=[-999, 0, 999], alpha=0.25, colors=['#ff9999', '#99ccff'])
    plt.contour(grid_x, grid_y, Z, levels=[0], colors='black', linewidths=2)
    for x, y in zip(xs, ys):
        color = '#1f77b4' if y == 1 else '#d62728'
        marker = 'o' if y == 1 else 'x'
        plt.scatter(x[0], x[1], c=color, marker=marker, s=80)
    plt.title('Mini project classifier')
    plt.xlabel('x1')
    plt.ylabel('x2')
    plt.grid(True, alpha=0.2)
    plt.show()
else:
    print('matplotlib 不可用，跳过画图。')

## 6. TODO Lab - 写 accuracy_from_scores

TODO：给普通数字 score 和 label，算准确率。

In [ ]:
def accuracy_from_scores(scores, labels):
    # TODO: score > 0 预测 1，否则预测 -1；返回 correct / len(labels)
    pass


def qa_check_10_accuracy(func):
    out = func([0.5, -0.2, 3.0, -1.0], [1, -1, -1, -1])
    if out is None:
        print('请先填写 TODO：返回准确率。')
        return
    expected = 0.75
    assert abs(out - expected) < 1e-9, f'expected {expected}, got {out}'
    print('OK: accuracy_from_scores passed.')


qa_check_10_accuracy(accuracy_from_scores)

<details>
<summary><strong>Show / Hide 提示</strong></summary>

先把每个 score 变成预测标签：

```python
pred = 1 if score > 0 else -1
```

然后数 `pred == label` 的数量。

</details>

<details>
<summary><strong>Show / Hide 答案</strong></summary>

```python
def accuracy_from_scores(scores, labels):
    preds = [1 if score > 0 else -1 for score in scores]
    correct = sum(int(pred == label) for pred, label in zip(preds, labels))
    return correct / len(labels)
```

</details>

## 7. 项目复盘模板

学完后，用这 5 行写在自己的笔记里：

```text
我的数据是什么：
我的模型是什么：
我的 loss 是什么：
训练前后 loss/accuracy 怎么变：
如果结果不好，我会先检查：
```

## What To Remember

```text
1. 一个项目不是只有 model，还包括数据、loss、训练、评估和解释。
2. score 是模型输出，label 是目标答案。
3. loss 用来训练，accuracy 用来评价分类结果。
4. loss.backward() 只算梯度，真正改变参数的是 update。
5. 能解释整条链路，比只看到 loss 下降更重要。
```

<!-- xiao-post:start -->
## 课后作业提交清单

```text
1. 我跑完整个 mini project，没有跳过 TODO。
2. 我能解释数据、模型、loss、accuracy 分别是什么。
3. 我能说明训练循环每一行的作用。
4. 我能看懂最终图里的 decision boundary。
5. 我能说出如果 loss 不下降，我会先检查什么。
```

## AI 复盘检查 Prompt

```text
你是我的 micrograd mini project 答辩官。
我刚完成一个 tiny classifier 项目。
请你一次只问一个问题，检查我是否能解释：
1. 数据 xs/ys 的含义
2. MLP(2, [4, 1]) 的结构
3. loss 和 accuracy 的区别
4. backward 和 update 分别做什么
5. decision boundary 图代表什么
6. 如果 loss 不下降，我应该怎么排查
最后请给我一个 0-100 的项目掌握度评分，并判断我是否可以开始学 PyTorch 或更大的深度学习项目。
```